In [4]:
"""
Benchmark 1: Clinical Cox Model (Baseline)

ROOT CAUSE NOTE (same class of bug as Benchmark 3):
The original code iterated all visits for the patient, including
post-conversion AD visits. For Benchmark 1 this matters for the
longitudinal rows exported to R: the LME/JM models in R should only
see MCI-era MMSE trajectories, not post-conversion MMSE which is
confounded with the outcome.

FIXES:
- [CRITICAL] Exported visits restricted to MCI-onwards, pre-conversion
  rows only (df.iloc[mci_idx:seq_end_idx]). R's LME will now model the
  MCI-era MMSE trajectory only, not post-conversion decline.
- [CRITICAL] dx_progression removed from TEMPORAL_FEATURES — it encodes
  the outcome directly (MCI->AD transition = +1).
- [CRITICAL] visit_number replaced with time_since_mci.
- [FIX]      adas_mmse_discordance uses past-only statistics (row-by-row).
- [MINOR]    TEMPORAL_FEATURES inlined (no .extend() on re-import).
- Retained: VALID_PATIENTS gate, MCI-anchored survival time.
"""

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# FEATURE DEFINITIONS
# dx_progression REMOVED (direct outcome encoding).
# visit_number REPLACED by time_since_mci.
# ============================================================================

STATIC_FEATURES = [
    'age_bl', 'PTGENDER_encoded', 'PTEDUCAT', 'PTETHCAT_encoded',
    'PTRACCAT_encoded', 'PTMARRY_encoded'
]

TEMPORAL_FEATURES = [
    'time_since_mci',
    'time_from_baseline',
    'AGE', 'age_since_bl',
    'mmse_slope', 'adas13_slope',
    # dx_progression REMOVED
    'cog_decline_index',
    'MMSE', 'ADAS13',
    'age_mmse_interaction', 'education_cognitive_reserve',
    'rapid_decline_flag', 'mmse_severity',
    'weighted_mmse_decline', 'mmse_variability',
    'adas_mmse_discordance'
]

# ============================================================================
# FEATURE ENGINEERING
# Called on MCI-onwards slice only — all features are already restricted
# to the correct time window before this function runs.
# ============================================================================

def engineer_features(df, mci_time):
    df = df.sort_values("Years_bl").reset_index(drop=True)

    df["time_from_baseline"] = df["Years_bl"] - df["Years_bl"].iloc[0]
    df["time_since_mci"]     = df["Years_bl"] - mci_time
    df["age_bl"]             = df["AGE"].iloc[0]
    df["age_since_bl"]       = df["AGE"] - df["age_bl"]

    dt = df["Years_bl"].diff()
    df["mmse_slope"]   = df["MMSE"].diff()   / dt
    df["adas13_slope"] = df["ADAS13"].diff() / dt

    df["cog_decline_index"] = df["ADAS13"] - df["MMSE"]

    df["age_mmse_interaction"]        = df["AGE"] * (30 - df["MMSE"]) / 30
    df["education_cognitive_reserve"] = df["PTEDUCAT"] * df["MMSE"] / 30
    df["rapid_decline_flag"]          = (df["mmse_slope"] < -2).astype(float)

    mmse_bins = [0, 20, 24, 30]
    df["mmse_severity"] = pd.cut(
        df["MMSE"], bins=mmse_bins, labels=[2, 1, 0]
    ).astype(float)

    df["weighted_mmse_decline"] = (
        df["mmse_slope"] * np.exp(-0.1 * df["time_since_mci"])
    )
    df["mmse_variability"] = df["MMSE"].rolling(window=3, min_periods=1).std()

    # Row-by-row discordance — past visits only
    adas_disc = np.zeros(len(df))
    for t in range(len(df)):
        past_adas = df["ADAS13"].iloc[:t + 1]
        past_mmse = df["MMSE"].iloc[:t + 1]
        adas_mean = past_adas.mean()
        adas_std  = past_adas.std() if len(past_adas) > 1 else 0.0
        mmse_mean = past_mmse.mean()
        mmse_std  = past_mmse.std() if len(past_mmse) > 1 else 0.0
        adas_z = (df["ADAS13"].iloc[t] - adas_mean) / (adas_std + 1e-7)
        mmse_z = (df["MMSE"].iloc[t]   - mmse_mean) / (mmse_std + 1e-7)
        adas_disc[t] = abs(adas_z - mmse_z)

    df["adas_mmse_discordance"] = adas_disc
    df = df.fillna(0)
    return df


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 80)
    print("BENCHMARK 1: CLINICAL COX MODEL (BASELINE)")
    print("=" * 80)

    print("\nLoading valid patient list...")
    with open('VALID_PATIENTS.pkl', 'rb') as f:
        VALID_PATIENTS = pickle.load(f)
    VALID_PATIENTS = set(str(p) for p in VALID_PATIENTS)
    print(f"Valid patients: {len(VALID_PATIENTS)}")

    manifest = pd.read_csv("./AD_Multimodal/TFN_AD/AD_Patient_Manifest.csv")
    manifest["PTID"] = manifest["PTID"].astype(str)
    manifest["path"] = manifest["path"].str.replace("\\", "/", regex=False)
    manifest["path"] = "./AD_Multimodal/TFN_AD/" + manifest["path"]

    all_rows  = []
    processed = 0
    skipped   = 0

    print("\nProcessing patients...")
    for ptid in manifest["PTID"].unique():
        if ptid not in VALID_PATIENTS:
            skipped += 1
            continue

        try:
            patient_rows = manifest[manifest["PTID"] == ptid]
            df_full = pd.read_pickle(patient_rows.iloc[0]["path"])
            df_full = df_full.sort_values("Years_bl").reset_index(drop=True)

            dx_seq = df_full["DX"].tolist()
            if "MCI" not in dx_seq:
                continue

            mci_idx  = dx_seq.index("MCI")
            mci_time = float(df_full["Years_bl"].iloc[mci_idx])

            ad_idx = next(
                (i for i, x in enumerate(dx_seq[mci_idx + 1:],
                                         start=mci_idx + 1)
                 if x in ["AD", "Dementia"]),
                -1
            )

            if ad_idx != -1:
                time_to_event = (df_full["Years_bl"].iloc[ad_idx] - mci_time)
                event         = 1
                seq_end_idx   = ad_idx          # stop BEFORE conversion visit
            else:
                time_to_event = (df_full["Years_bl"].iloc[-1] - mci_time)
                event         = 0
                seq_end_idx   = len(df_full)

            # Slice to MCI-onwards, pre-conversion visits
            df_mci = df_full.iloc[mci_idx:seq_end_idx].copy()

            if len(df_mci) < 2:
                continue

            df_mci = engineer_features(df_mci, mci_time)

            for _, visit in df_mci.iterrows():
                row = {
                    "PTID":          ptid,
                    "Years_bl":      visit["Years_bl"],
                    "MMSE":          visit["MMSE"],
                    "ADAS13":        visit["ADAS13"],
                    "time_to_event": time_to_event,
                    "event":         event,
                    "AGE":           visit["AGE"],
                    "PTGENDER":      visit.get("PTGENDER_encoded", 0),
                    "PTEDUCAT":      visit["PTEDUCAT"],
                }
                for feat in TEMPORAL_FEATURES + STATIC_FEATURES:
                    if feat in df_mci.columns:
                        row[feat] = visit[feat]
                all_rows.append(row)

            processed += 1

        except Exception as e:
            print(f"  Warning: skipped patient {ptid}: {e}")
            continue

    df_out = pd.DataFrame(all_rows).sort_values(["PTID", "Years_bl"])
    df_out.to_csv("baseline_clinical_features.csv", index=False)

    n_patients = df_out["PTID"].nunique()
    n_events   = df_out.groupby("PTID")["event"].first().sum()

    print("\n" + "=" * 80)
    print("CLINICAL FEATURES EXTRACTED")
    print("=" * 80)
    print(f"  Processed : {processed}")
    print(f"  Skipped   : {skipped} (not in valid set)")
    print(f"  Patients  : {n_patients}")
    print(f"  Visits    : {len(df_out)}")
    print(f"  Events    : {n_events} ({100 * n_events / n_patients:.1f}%)")
    print("  Visits    : MCI-onwards, pre-conversion only")
    print("  dx_progression removed from features")
    print("=" * 80)


if __name__ == "__main__":
    main()

BENCHMARK 1: CLINICAL COX MODEL (BASELINE)

Loading valid patient list...
Valid patients: 161

Processing patients...

CLINICAL FEATURES EXTRACTED
  Processed : 147
  Skipped   : 221 (not in valid set)
  Patients  : 147
  Visits    : 694
  Events    : 65 (44.2%)
  Visits    : MCI-onwards, pre-conversion only
  dx_progression removed from features
